## Célula 0: Imports Globais

Rode esta célula primeiro para carregar todas as bibliotecas necessárias


In [1]:
import json
import time
import random
import shutil
from pathlib import Path

print("✅ Todas as bibliotecas carregadas com sucesso!")

✅ Todas as bibliotecas carregadas com sucesso!


## Célula 1: Configurações Iniciais

Rode esta célula para configurar os caminhos e as classes alvo

In [2]:
BASE_DIR = Path.cwd() 
COCO_DIR = BASE_DIR / "coco_2017"
ANNOTATIONS_FILE = COCO_DIR / "annotations" / "instances_train2017.json"
IMAGES_DIR = COCO_DIR / "train2017"

OUTPUT_DIR = BASE_DIR / "FasterRCNN_dataset"
OUTPUT_IMAGES = OUTPUT_DIR / "images"
OUTPUT_ANNOTATIONS = OUTPUT_DIR / "annotations"

IMAGENS_POR_CLASSE = 3000

COCO_CLASSES = {
    1: 'person',
    2: 'bicycle',
    3: 'car',
    4: 'motorcycle',
    6: 'bus'       
}

print(f"✅ Diretório base configurado para: {BASE_DIR}")
print(f"🎯 Meta: {IMAGENS_POR_CLASSE} imagens por classe alvo.")

✅ Diretório base configurado para: /home/rcferreira/Documents/TCC/COCO-2017
🎯 Meta: 3000 imagens por classe alvo.


## Célula 2: Funções de Suporte

Rode esta célula para registrar as funções na memória do Jupyter

In [3]:
def setup_directories():
    if OUTPUT_DIR.exists():
        print("🧹 Limpando diretório antigo...")
        shutil.rmtree(OUTPUT_DIR)
        
    OUTPUT_IMAGES.mkdir(parents=True, exist_ok=True)
    OUTPUT_ANNOTATIONS.mkdir(parents=True, exist_ok=True)
    
    # Criar subpastas de imagens previamente
    for split in ['train', 'val', 'test']:
        (OUTPUT_IMAGES / split).mkdir(exist_ok=True)
        
    print(f"📁 Diretórios criados em: {OUTPUT_DIR}")

print("✅ Funções auxiliares carregadas!")

✅ Funções auxiliares carregadas!


## Célula 3: O Algoritmo de Mineração (O Cérebro do Script)

Rode esta célula. É aqui que o JSON é lido e a matemática de balanceamento e sobreposição de imagens acontece na RAM. Nenhuma foto é copiada ainda.

In [4]:
# ==========================================
# 3. LEITURA E BALANCEAMENTO LÓGICO
# ==========================================

start_time = time.time()
print("⏳ Carregando o arquivo JSON do COCO... (Pode levar 1-2 minutos dependendo da sua RAM)")

with open(ANNOTATIONS_FILE, 'r') as f:
    coco_data = json.load(f)
print("✅ JSON carregado na memória!")

# Mapeamento rápido de imagens
images_info = {img['id']: img for img in coco_data['images']}

# Agrupa anotações por imagem (filtrando multidões borradas e focando nas 5 classes)
annotations_by_image = {}
for ann in coco_data['annotations']:
    cat_id = ann['category_id']
    if cat_id in COCO_CLASSES and not ann.get('iscrowd', 0):
        img_id = ann['image_id']
        if img_id not in annotations_by_image:
            annotations_by_image[img_id] = []
        annotations_by_image[img_id].append(ann)

# Dicionários de rastreamento para o nosso relatório final
selected_image_ids = set()
images_per_class = {cat_id: 0 for cat_id in COCO_CLASSES}
objects_per_class = {cat_id: 0 for cat_id in COCO_CLASSES}

print(f"🎯 Iniciando a busca exata de {IMAGENS_POR_CLASSE} imagens por classe...")

# Abordagem de preenchimento (Greedy)
for target_cat_id, class_name in COCO_CLASSES.items():
    for img_id, anns in annotations_by_image.items():
        if images_per_class[target_cat_id] >= IMAGENS_POR_CLASSE:
            break # Bateu a meta dessa classe, vai pra próxima
            
        has_target_class = any(a['category_id'] == target_cat_id for a in anns)
        
        if has_target_class and img_id not in selected_image_ids:
            selected_image_ids.add(img_id)
            
            # Conta os objetos físicos dentro dessa foto e atualiza o rastreador de imagens
            classes_in_this_image = set()
            for a in anns:
                c_id = a['category_id']
                if c_id in COCO_CLASSES:
                    objects_per_class[c_id] += 1
                    classes_in_this_image.add(c_id)
                    
            for c_id in classes_in_this_image:
                images_per_class[c_id] += 1

print(f"\n🔥 Mineração concluída em {time.time() - start_time:.2f} segundos!")
print(f"Total de imagens únicas selecionadas para extração: {len(selected_image_ids)}")

⏳ Carregando o arquivo JSON do COCO... (Pode levar 1-2 minutos dependendo da sua RAM)
✅ JSON carregado na memória!
🎯 Iniciando a busca exata de 3000 imagens por classe...

🔥 Mineração concluída em 11.61 segundos!
Total de imagens únicas selecionadas para extração: 10332


### Célula 4: Fatiamento, Extração Física e Geração dos JSONs
Nesta versão, unificamos a extração e o fatiamento (antigas células 4 e 6). Como o Faster R-CNN precisa de um arquivo JSON por etapa, faz sentido dividir a lista de IDs primeiro, copiar as imagens para suas respectivas pastas e gerar os três arquivos (train.json, val.json, test.json) de uma só vez.

In [5]:
# =======================================================
# 4. SPLIT E GERAÇÃO DOS ARQUIVOS JSON (PADRÃO COCO)
# =======================================================

print("⚙️ Preparando diretórios de saída...")
setup_directories()

# Define a semente matemática e cria os cortes
random.seed(42)
lista_ids = list(selected_image_ids)
random.shuffle(lista_ids)

total = len(lista_ids)
train_idx = int(total * 0.80)
val_idx = int(total * 0.90)

splits = {
    "train": lista_ids[:train_idx],
    "val": lista_ids[train_idx:val_idx],
    "test": lista_ids[val_idx:]
}

print(f"\n🧮 Total de imagens para dividir: {total}")
print(f"   ┣ 🏋️ Treino (80%): {len(splits['train'])}")
print(f"   ┣ 📏 Validação (10%): {len(splits['val'])}")
print(f"   ┗ 🎯 Teste (10%): {len(splits['test'])}\n")

# Prepara a estrutura base do JSON COCO
categorias_coco = [{"id": k, "name": v, "supercategory": "none"} for k, v in COCO_CLASSES.items()]

imgs_copiadas = 0

print("🚀 Iniciando a cópia física e geração dos JSONs (Aguarde...)")

for split_name, img_ids in splits.items():
    
    coco_format = {
        "info": {"description": f"Dataset Faster R-CNN - {split_name}"},
        "images": [],
        "annotations": [],
        "categories": categorias_coco
    }
    
    ann_id_counter = 1
    
    for img_id in img_ids:
        img_data = images_info[img_id]
        file_name = img_data['file_name']
        
        src_img_path = IMAGES_DIR / file_name
        dst_img_path = OUTPUT_IMAGES / split_name / file_name
        
        if src_img_path.exists():
            shutil.copy2(src_img_path, dst_img_path)
            imgs_copiadas += 1
            
            # Adiciona metadados da imagem ao JSON
            coco_format["images"].append({
                "id": img_id,
                "file_name": file_name,
                "width": img_data['width'],
                "height": img_data['height']
            })
            
            # Adiciona anotações filtradas
            for ann in annotations_by_image[img_id]:
                if ann['category_id'] in COCO_CLASSES:
                    ann_copy = ann.copy()
                    ann_copy['id'] = ann_id_counter 
                    coco_format["annotations"].append(ann_copy)
                    ann_id_counter += 1
                    
    # Salva o JSON específico deste split
    json_path = OUTPUT_ANNOTATIONS / f"{split_name}.json"
    with open(json_path, 'w') as f:
        json.dump(coco_format, f)

print(f"✅ Fim da extração! {imgs_copiadas} imagens copiadas organizadas nas pastas com seus respectivos JSONs.")

⚙️ Preparando diretórios de saída...
📁 Diretórios criados em: /home/rcferreira/Documents/TCC/COCO-2017/FasterRCNN_dataset

🧮 Total de imagens para dividir: 10332
   ┣ 🏋️ Treino (80%): 8265
   ┣ 📏 Validação (10%): 1033
   ┗ 🎯 Teste (10%): 1034

🚀 Iniciando a cópia física e geração dos JSONs (Aguarde...)
✅ Fim da extração! 10332 imagens copiadas organizadas nas pastas com seus respectivos JSONs.


### Célula 5: Relatório Final
Alterada apenas para refletir os rótulos do COCO sem o YOLO_MAPPING

In [6]:
print("\n" + "="*95)
print("📊 ANÁLISE DE DISTRIBUIÇÃO DE CLASSES (FASTER R-CNN BALANCEADO)")
print("="*95)
print(f"{'Classe ID':<10} {'Nome da Classe':<15} {'Total de Objetos':<20} {'Total de Imagens':<20} {'Média de Objetos/Imagem'}")
print("-" * 95)

total_objetos_geral = 0
for cat_id, name in COCO_CLASSES.items():
    total_obj = objects_per_class[cat_id]
    total_img = images_per_class[cat_id]
    media = total_obj / total_img if total_img > 0 else 0
    total_objetos_geral += total_obj
    
    print(f"{cat_id:<10} {name:<15} {total_obj:<20} {total_img:<20} {media:.2f}")

print("="*95)
print("\n📈 RESUMO GERAL")
print("="*95)
print(f"✓ Total de imagens processadas fisicamente: {imgs_copiadas}")
print(f"✓ Total de objetos detectados (Bounding Boxes): {total_objetos_geral}")

media_geral = total_objetos_geral / imgs_copiadas if imgs_copiadas > 0 else 0
print(f"✓ Média de objetos por imagem no Dataset: {media_geral:.2f}")
print(f"✓ Número de classes ativas: {len(COCO_CLASSES)}")
print("="*95)
print(f"\n🎉 Tudo pronto! Pode zipar a pasta '{OUTPUT_DIR.name}'.")


📊 ANÁLISE DE DISTRIBUIÇÃO DE CLASSES (FASTER R-CNN BALANCEADO)
Classe ID  Nome da Classe  Total de Objetos     Total de Imagens     Média de Objetos/Imagem
-----------------------------------------------------------------------------------------------
1          person          41776                8086                 5.17
2          bicycle         6615                 3000                 2.21
3          car             17161                4652                 3.69
4          motorcycle      7172                 3023                 2.37
6          bus             4538                 3000                 1.51

📈 RESUMO GERAL
✓ Total de imagens processadas fisicamente: 10332
✓ Total de objetos detectados (Bounding Boxes): 77262
✓ Média de objetos por imagem no Dataset: 7.48
✓ Número de classes ativas: 5

🎉 Tudo pronto! Pode zipar a pasta 'FasterRCNN_dataset'.


### Célula 6: Compactação
Apenas ajustes nas variáveis de diretório.

In [7]:
DATASET_DIR = BASE_DIR / "FasterRCNN_dataset"  
ZIP_OUTPUT = BASE_DIR / "FasterRCNN_dataset"  

if DATASET_DIR.exists():
    print(f"📦 Iniciando a compactação da pasta '{DATASET_DIR.name}'...")
    start_time = time.time()
    
    shutil.make_archive(ZIP_OUTPUT, 'zip', DATASET_DIR)
    
    print("\n" + "="*75)
    print(f"✅ SUCESSO! Arquivo '{ZIP_OUTPUT.name}.zip' gerado.")
    print(f"📍 Localização: {BASE_DIR / (ZIP_OUTPUT.name + '.zip')}")
    print(f"⏱️  Tempo de execução: {time.time() - start_time:.2f} segundos.")
    print("="*75)
else:
    print(f"❌ Erro: O diretório '{DATASET_DIR}' não foi encontrado.")

📦 Iniciando a compactação da pasta 'FasterRCNN_dataset'...

✅ SUCESSO! Arquivo 'FasterRCNN_dataset.zip' gerado.
📍 Localização: /home/rcferreira/Documents/TCC/COCO-2017/FasterRCNN_dataset.zip
⏱️  Tempo de execução: 48.65 segundos.
